In this notebook, we will investigate the ART matrix and see if we can find a pressure domain PU matrix that matches the ART energy matrix, $\mathbf{S}$ exactly. The ART matrix is column-stochastic, i.e, 
$$
\begin{aligned}
\sum_h S_{h\to i \to j} &= 1 \quad \forall\, i,j \quad \text{(lossless)}, \\
\mathbf{1}^\top \mathbf{S} &= \mathbf{1}^\top.
\end{aligned}
$$

Now, we want to find a PU matrix, $\mathbf{H}(e^{j\omega})$ such that
$$\frac{1}{2 \pi} \int_0^{2\pi}|H_{ij}(e^{j\omega})|^2 d\omega = S_{ij}, \quad \mathbf{H}(e^{j\omega})\mathbf{H}(e^{-j\omega})^\top = \mathbf{I}$$

Note that $|\mathbf{H}(e^{j\omega})|^{\odot 2}$ is doubly stochastic. However, $\mathbf{S}$ is only column-stochastic. For the above equivalence to hold, $\mathbf{S}$ must be made doubly stochastic using the Sinkhorn-Knopp algorthm. However, for this $\mathbf{S}$ requires total support but that is unlikely due to the sparsity of the matrix.

1. First, we prune the matrix $\mathbf{S}$ to remove connections of patches that are not visible to each other. For an $\mathbf{S} \in \mathbb{R}^{+(N \times N)}$, if $k$ patches are not visible to each other then the matrix is pruned to be of size, $\mathbf{S}_{p} \in \mathbb{R}^{+(N-k) \times (N-k)}$.
2.  We run Sinkhorn-Knopp on the pruned matrix to get a doubly stochastic matrix, $\hat{\mathbf{S}}_p, \text{ s.t. } \mathbf{1}^\top\hat{\mathbf{S}}_p = \mathbf{1}^T,  \hat{\mathbf{S}}_p \mathbf{1} = \mathbf{1}.$
3.  We find the FIR PU matrix, $\mathbf{H}(z)$, s.t, $\int_{0}^{2\pi} |\mathbf{H}(e^{j\omega})|^{\odot 2}d \omega = \hat{\mathbf{S}}_p$.
4.  We operate TD-ART in the pressure domain with the FIR PU matrix.

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
import scipy
import os
from pathlib import Path
from loguru import logger

from slope2noise.utils import db
import SDNPy.utils.ARTMatrixMath as mm
import SDNPy.utils.visualize_room as vis
from SDNPy.DataTypes import DelayConstructionType
from SDNPy.utils.MatrixMath import closestSignAgnosticOrthonormal, sinkhornKnopp

from raves.src.utils.raves_io import load_all_inputs

In [ ]:
patch_area = 2.0
environment_name = f'ERTD1_generated_patch_area={patch_area:.1f}'
# environment_name = 'ERTD2_1_patch_per_wall'
environment_folder = os.path.join('..', 'environment', environment_name)
sample_rate = 44100

fig_path = Path(f'../../../Figures/ART/ERTD/matrices/{environment_name}')
fig_path.mkdir(parents=True, exist_ok=True)

#### Set up logging

In [ ]:
logs_dir = Path(f"{fig_path}/logs")
logs_dir.mkdir(parents=True, exist_ok=True)
log_path = logs_dir / f"ertd_doubly_stochastic_to_paraunitary_logger.txt"
if log_path.exists():
    log_path.unlink()
logger.add(log_path, enqueue=True, backtrace=False, diagnose=False)

#### Read ART patching matrix and reflection matrix

In [ ]:
# Read the .mtx file
art_reflect_matrix = scipy.io.mmread(f'{environment_folder}/ART_kernel_band_1.mtx')
art_diffuse_matrix = scipy.io.mmread(f'{environment_folder}/ART_kernel_diffuse.mtx')
art_specular_matrix = scipy.io.mmread(f'{environment_folder}/ART_kernel_specular.mtx')
art_patching_matrix = scipy.io.mmread(f'{environment_folder}/path_indexing.mtx')

# Convert to dense format if needed for visualization (optional)
art_full_dense_matrix = art_reflect_matrix.toarray()
art_diffuse_dense_matrix = art_diffuse_matrix.toarray()
art_specular_dense_matrix = art_specular_matrix.toarray()
art_patching_matrix_dense = art_patching_matrix.toarray()

num_non_zero_elems = np.count_nonzero(art_patching_matrix_dense)
assert art_full_dense_matrix.shape[0] == num_non_zero_elems

# Extract patch labels in mesh order from mesh.obj
mesh_obj_path = Path(environment_folder) / 'mesh.obj'
patch_labels = []
with open(mesh_obj_path, 'r', encoding='utf-8') as f:
    for line in f:
        line_no_comment = line.split('#', 1)[0].strip()
        if not line_no_comment.startswith('usemtl '):
            continue
        mat_name = line_no_comment.split()[1]
        if mat_name not in patch_labels:
            patch_labels.append(mat_name)

num_patches = art_patching_matrix_dense.shape[0]
assert len(patch_labels) == num_patches, (
    f'Expected {num_patches} patch labels from mesh.obj, found {len(patch_labels)}.'
)

#### Plot patching matrix - out of 256 total connections, 134 are valid

In [ ]:
# Create a Spy plot to visualize sparsity
plt.figure(figsize=(9, 8))
plt.spy(art_patching_matrix, markersize=2)
plt.title('Patching Matrix Sparsity Pattern')
plt.xlabel('Patch #')
plt.ylabel('Patch #')
plt.xticks(range(num_patches), patch_labels, rotation=90, fontsize=7)
plt.yticks(range(num_patches), patch_labels, fontsize=7)
plt.tight_layout()
plt.savefig(f'{fig_path.resolve()}/{environment_name}_patching_matrix.png')
plt.show()

#### Plot the sparse ART diffuse and specular matrices and their pruned, dense version for each patch

In [ ]:
label = ['diffuse', 'specular']
for k, art_dense_matrix in enumerate([art_diffuse_dense_matrix, art_specular_dense_matrix]):
    # Create a Heatmap for density
    plt.figure(figsize=(12, 12))
    plt.imshow(art_dense_matrix, cmap='viridis', interpolation='nearest')
    plt.colorbar()
    plt.title(f'{label[k]} ART matrix')
    plt.xlabel('Out. chans (idx of ij)')
    plt.ylabel('Inc. chans (idx of hi)')
    plt.show()

    num_subplots = min(6, np.round(np.sqrt(num_patches)).astype(int))
    art_matrix_pruned_list = mm.prune_tdart_matrix(art_dense_matrix, art_patching_matrix.copy())
    fig, axes = plt.subplots(num_subplots, num_subplots, figsize=(12, 12), constrained_layout=True)
    for i, ax in enumerate(axes.flat):
        # make sure no zero elements in the matrices
        num_elems =  art_matrix_pruned_list[i].shape[0] * art_matrix_pruned_list[i].shape[1]
        ax.imshow(art_matrix_pruned_list[i], cmap='viridis', interpolation='nearest')
        # ax.set_title(f"{patch_labels[i]}", fontsize=8)
        # ax.set_xlabel("i -> j")
        # ax.set_ylabel("h -> i")

    if k == 0:
        art_diffuse_matrix_pruned_list = art_matrix_pruned_list.copy()
    elif k == 1:
        art_specular_matrix_pruned_list = art_matrix_pruned_list.copy()
        
    plt.savefig(f'{fig_path.resolve()}/{environment_name}_stochastic_{label[k]}_art_matrices.png')    
    plt.show()

#### Read mesh and patch properties to create proper ART reflection matrix without absorption

In [ ]:
mesh, patch_materials, material_coefficients, environment_folder = load_all_inputs(environment_folder, area_threshold=0, thoroughness=0)
assert len(patch_materials) == num_patches, "Number of patch materials must match number of patches"
patch_absorption = []
patch_scattering = []
art_matrix_pruned_list = []

for band_idx, center_frequency in enumerate(material_coefficients['Frequencies']):

    for i, patch_mat in enumerate(patch_materials):
        # Retrieve the coefficients of patch i for this frequency band.
        patch_absorption.append(material_coefficients[patch_mat][0, band_idx])
        patch_scattering.append(material_coefficients[patch_mat][1, band_idx])

        art_matrix_pruned_list.append(art_diffuse_matrix_pruned_list[i] * patch_scattering[i] + 
                                      art_specular_matrix_pruned_list[i] * (1 - patch_scattering[i]))


save_path = Path(environment_folder) / f'ART_kernel_band_{band_idx+1}_no_absorp.mtx'
art_ss_matrix_full = mm.reconstruct_full_tdart_matrix(art_matrix_pruned_list, art_patching_matrix_dense, 
                                                      [0.0 for k in range(len(patch_absorption))], 
                                                      save_path=save_path, operate_in_pressure_domain=False)
plt.figure(figsize=(12, 12))
plt.imshow(art_ss_matrix_full, cmap='viridis', interpolation='nearest')
plt.colorbar()
plt.title(f'Full ART matrix')
plt.xlabel('Out. chans (idx of ij)')
plt.ylabel('Inc. chans (idx of hi)')
plt.savefig(f'{fig_path.resolve()}/{environment_name}_stochastic_full_art_matrix.png')    
plt.show()


col_sum = art_ss_matrix_full.sum(axis=0)
row_sum = art_ss_matrix_full.sum(axis=1)
sum_matrix = np.outer(row_sum, col_sum)
plt.figure(figsize=(12, 12))
plt.imshow(db(sum_matrix + 1e-12), cmap='viridis', vmin = -60, vmax=20, interpolation='nearest')
plt.colorbar(label='db')
plt.title(f'Row*col sum of ART matrix')
plt.xlabel('Col #')
plt.ylabel('Row #')
plt.savefig(f'{fig_path.resolve()}/{environment_name}_rowxcol_sum_full_art_matrix.png')    
plt.show()

In [ ]:
fig1, axes = plt.subplots(num_subplots, num_subplots, figsize=(12, 12), constrained_layout=True)
fig2, axes2 = plt.subplots(num_subplots, num_subplots, figsize=(12, 12), constrained_layout=True)

for i, (ax1, ax2) in enumerate(zip(axes.flat, axes2.flat)):
    # make sure no zero elements in the matrices
    num_elems =  art_matrix_pruned_list[i].shape[0] * art_matrix_pruned_list[i].shape[1]
    col_sum = art_matrix_pruned_list[i].sum(axis=0)
    row_sum = art_matrix_pruned_list[i].sum(axis=1)
    sum_matrix = np.outer(row_sum, col_sum)
    
    assert np.count_nonzero(art_matrix_pruned_list[i]) == num_elems    
    im1 = ax1.imshow(art_matrix_pruned_list[i], cmap='viridis', interpolation='nearest')
    im2 = ax2.imshow(db(sum_matrix), cmap='viridis', interpolation='nearest', vmin = -60, vmax=20.0)
    
    # ax.set_title(f"{patch_labels[i]}", fontsize=8)
    # ax.set_xlabel("i -> j")
    # ax.set_ylabel("h -> i")
    
fig1.savefig(f'{fig_path.resolve()}/{environment_name}_stochastic_art_matrices.png')
fig2.colorbar(im2, ax=axes2.ravel(), label = 'dB', orientation='vertical')
fig2.savefig(f'{fig_path.resolve()}/{environment_name}_rowxcol_sum_art_matrices.png')
plt.show()

#### Plot the doubly stochastic Sinkhorn Knopp versions of the smaller matrices

In [ ]:
fig, axes = plt.subplots(num_subplots, num_subplots, figsize=(12, 12), constrained_layout=True)
art_matrix_pruned_double_stochastic = []

art_matrix_pruned_double_stochastic = [sinkhornKnopp(art_matrix_pruned, verbose=False) for art_matrix_pruned in art_matrix_pruned_list]

for i, ax in enumerate(axes.flat):
    try:
        cur_double_stochastic = sinkhornKnopp(art_matrix_pruned_list[i], verbose=False)
    except AssertionError as e:
        logger.error("Sinkhorn Knopp failed")
    assert mm.is_doubly_stochastic(cur_double_stochastic)
    ax.imshow(cur_double_stochastic, cmap='viridis', interpolation='nearest')
    # ax.set_title(f"{patch_labels[i]}", fontsize=8)
    # ax.set_xlabel("i -> j")
    # ax.set_ylabel("h -> i")

plt.savefig(f'{fig_path.resolve()}/{environment_name}_doubly_stochastic_art_matrices.png')    
plt.show()

#### Save the Sinkhorn-Knopp generated DS matrix

In [ ]:
save_folder = Path(f'{environment_folder}_doubly_stochastic_reflection_matrix/')
save_folder.mkdir(parents=True, exist_ok=True)
save_path = save_folder / f'ART_kernel_band_{band_idx+1}.mtx'
art_ds_matrix_full = mm.reconstruct_full_tdart_matrix(art_matrix_pruned_double_stochastic, art_patching_matrix_dense, 
                                                      patch_absorption, save_path=save_path, operate_in_pressure_domain=False)

plt.figure(figsize=(12, 12))
plt.imshow(art_ds_matrix_full, cmap='viridis', interpolation='nearest')
plt.colorbar()
plt.title(f'Full DS ART matrix')
plt.xlabel('Out. chans (idx of ij)')
plt.ylabel('Inc. chans (idx of hi)')
plt.savefig(f'{fig_path.resolve()}/{environment_name}_doubly_stochastic_full_art_matrix.png')    
plt.show()

art_ds_matrix_pruned_list = mm.prune_tdart_matrix(art_ds_matrix_full, art_patching_matrix.copy())
fig, axes = plt.subplots(num_subplots, num_subplots, figsize=(12, 12), constrained_layout=True)
for i, ax in enumerate(axes.flat):
    # make sure no zero elements in the matrices
    num_elems =  art_ds_matrix_pruned_list[i].shape[0] * art_matrix_pruned_list[i].shape[1]
    ax.imshow(art_ds_matrix_pruned_list[i], cmap='viridis', interpolation='nearest')
    # ax.set_title(f"DS {patch_labels[i]}", fontsize=8)
    # ax.set_xlabel("i -> j")
    # ax.set_ylabel("h -> i")
plt.show()

#### Find the FIR PU matrix whose average power gives the desired doubly stochastic matrix

This is the most challenging part, 
   - We do a decomposition:
   $$\hat{\mathbf{S}}_p = U_0^{\odot 2} \ T_1 T_2 \ldots T_K,$$ $$T_r = t_r I + (1-t_r) Q$$ where Q is a permutation matrix to exchange two elements of a vector, eg to exchange (i, j) elements, set $Q = I$ and then exchange ith and jth rows (or columns). $U_0$ is a unitary matrix that is the sign-agnostic Procustes solution to $\hat{\mathbf{S}}_p^{\odot 0.5}$.
i.e., $T_r$ is identity except on a $2 \times 2$ block, $\begin{bmatrix} t_r & 1-t_r \\ 1-t_r & t_r\end{bmatrix}, 0 \leq t_r \leq 1$.  Now, $T_r = |G_r|^{\odot 2}$ where $G_r$ is a Givens rotation matrix.
        - This decomposition is tricky. We fix $K$. Then we find $K=\binom{N}{2}$ unique possible permutations of the $2 \times 2$ block with $K$ unique $t_r$ values. We use non-linear least squares to find $$\arg \min_{t_1, \ldots, t_K} ||\hat{\mathbf{S}}_p - U_0^{\odot 2} \ T_1(t_1) T_2(t_2) \ldots T_K(t_K)||^2_F,$$ where $0 \leq t_r \leq 1$, but we must test for $K$ possible pairs of $(i, j)$ while constructing the T-transform matrix for each $r$.
        - We use n_restarts with different sets of initial $t_r$ and different permutations of $T_r$ to give our optimisation problem a better chance at converging.
   - Now, we can build $$\mathbf{H}(z) = U_0 D_0(z) \ G_1 D_1(z) G_2 D_2(z) \ldots G_K,$$ where $D_r(z) = \text{diag}(z^{d_{r,1}}, \ldots, z^{d_{r,N-k}})$.
   - Note that,$$\frac{1}{2 \pi} \int_0^{2\pi}|\mathbf{H}(e^{j\omega})|^{\odot 2} d\omega  = U_0^{\odot 2} \ 
|G_1|^{\odot 2} |G_2|^{\odot 2} \ldots |G_{K}|^{\odot 2} = \hat{\mathbf{S}}_p$$. This holds as long as the delays $d_{r, n}$ are all pairwise distinct.
   - This is because $H_{ij}(e^{j\omega}) = \sum_p \alpha_p e^{-j\omega \tau_p}$, where $\tau_p$ is the total accumulated delay along that path, i.e, $\tau_p = \sum_{k=1}^K \omega_{k, c_k(p)}$ and $\alpha_p$ is the product of the Givens coefficients along that path. Here, $c_k(p)$ is a path such that $c_0(p) = j$, and $c_K(p) = i$. Therefore,
     $$\frac{1}{2 \pi} \int_0^{2\pi}|H_{ij}(e^{j\omega})|^2 d\omega = \sum_p |\alpha_p|^2 + \sum_{p \neq q} \alpha_p \alpha_q \frac{1}{2 \pi} \int_{0}^{2\pi} e^{-j\omega(\tau_p - \tau_q)} d\omega.$$ The cross-terms vanish if $\tau_p \neq \tau_q$.
   - A sufficient condition is to choose $D_k(z) = \text{diag}(z^0, \ldots, z^{N-1})^{N^k}$. However, this scales exponentially $N$ and $k$ giving very long FIR PU matrices (computationally bad).
   - More generally, $d_{r, i} > \sum_{\ell=1}^{r-1} \max_{m} d_{\ell, m}$. This means: every delay used at stage r is larger than the largest possible total delay that could have accumulated from all previous stages.

#### Convert the doubly stochastic matrices to PU matrices - use non-linear least squares for T-transform factorisation

In [ ]:
fig, axes = plt.subplots(num_subplots, num_subplots, figsize=(12, 12), constrained_layout=True)
fig2, axes2 = plt.subplots(num_subplots, num_subplots, figsize=(12, 12), constrained_layout=True)

art_matrix_pu_list_nls = []
for i in range(len(art_matrix_pruned_double_stochastic)):
    try:
        cur_art_ds_matrix = art_matrix_pruned_double_stochastic[i].copy()
        cur_mat_num_rows = cur_art_ds_matrix.shape[0]
        start_time = time.perf_counter()
        cur_art_pu_matrix, info = mm.doubly_stochastic_to_paraunitary(cur_art_ds_matrix, 
                                                                      factor_tol=1e-6,
                                                                      factor_method='nls',
                                                                      n_stages=2,
                                                                      n_restarts=8,
                                                                      delay_mode=DelayConstructionType.EXACT_BASE_N,
                                                                      exact_delay_max_taps=4096,
                                                                      return_info=True, 
                                                                      use_sa_unitary_init=True,)
        end_time = time.perf_counter()
        execution_time = end_time - start_time
        ss_recons_error = np.linalg.norm(info["average_energy"] - art_matrix_pruned_list[i], ord='fro') / np.linalg.norm(art_matrix_pruned_list[i], ord='fro')
        art_matrix_pu_list_nls.append(cur_art_pu_matrix)
        if i < num_subplots**2:
            ax1, ax2 = axes.flat[i], axes2.flat[i]
            ax1.imshow(info["average_energy"], cmap='viridis', interpolation='nearest')
            # ax1.set_title(f"{patch_labels[i]}", fontsize=8)
            # ax1.set_xlabel("i -> j")
            # ax1.set_ylabel("h -> i")
    
            ax2.imshow(info["fitted_doubly_stochastic"], cmap='viridis', interpolation='nearest')
            # ax2.set_title(f"{patch_labels[i]}", fontsize=8)
            # ax2.set_xlabel("i -> j")
            # ax2.set_ylabel("h -> i")

        logger.info(f"Singly stochastic PU recons error for NLS: {db(ss_recons_error):4f} dB")
        logger.info(f"Doubly stochastic reconstruction error NLS: {db(info['factor_error']):.4f} dB")
        logger.info(f"PU reconstruction error NLS: {db(info['average_energy_error']):.4f} dB")
        logger.info(f"Time taken NLS: {execution_time:.3f}s")
        
        assert db(info['factor_error']) <= -6, "Doubly stochastic reconstriction failed"
        assert db(info['average_energy_error']) <= -6, "PU reconstruction failed"
        assert info["is_paraunitary"], "Resulting matrix not unitary"
    except AssertionError as e:
        logger.error(f"PU reconstruction failed for {patch_labels[i]} scattering matrix of size {cur_mat_num_rows} x {cur_mat_num_rows}")
        continue
    
    # plot polynomial matrix
    vis.plot_poly_mats_overlay([cur_art_pu_matrix], 
                               labels=[patch_labels[i]], 
                               fs=sample_rate, 
                               title=f"{patch_labels[i]}, DS recons err ={db(info['factor_error']):.4f}dB",
                               save_path = f'{fig_path.resolve()}/{environment_name}_pu_art_matrix_{patch_labels[i]}_nls_sa_init.png',
                              )


fig.savefig(f'{fig_path.resolve()}/{environment_name}_pu_nls_sa_init_avg_energy_art_matrices.png')
fig2.savefig(f'{fig_path.resolve()}/{environment_name}_fitted_ds_nls_sa_init_avg_energy_art_matrices.png')    

plt.show()


#### Convert the doubly stochastic matrices to PU matrices - use CGD for T-transform factorisation

In [ ]:
fig, axes = plt.subplots(num_subplots, num_subplots, figsize=(12, 12), constrained_layout=True)
fig2, axes2 = plt.subplots(num_subplots, num_subplots, figsize=(12, 12), constrained_layout=True)


art_matrix_pu_list_cgd = []
for i in range(len(art_matrix_pruned_double_stochastic)):
    try:
        cur_art_ds_matrix = art_matrix_pruned_double_stochastic[i].copy()
        cur_mat_num_rows = cur_art_ds_matrix.shape[0]
        start_time = time.perf_counter()

        cur_art_pu_matrix, info = mm.doubly_stochastic_to_paraunitary(cur_art_ds_matrix, 
                                                                      factor_tol=1e-6,
                                                                      factor_method='cgd',
                                                                      n_stages=2,
                                                                      n_restarts=8,
                                                                      delay_mode=DelayConstructionType.EXACT_BASE_N,
                                                                      exact_delay_max_taps=4096,
                                                                      return_info=True,
                                                                      use_sa_unitary_init=True,
                                                                     )
        end_time = time.perf_counter()
        execution_time = end_time - start_time
        ss_recons_error = np.linalg.norm(info["average_energy"] - art_matrix_pruned_list[i], ord='fro') / np.linalg.norm(art_matrix_pruned_list[i], ord='fro')
        art_matrix_pu_list_cgd.append(cur_art_pu_matrix)        
        
        if i < num_subplots**2:
            ax1, ax2 = axes.flat[i], axes2.flat[i]
            ax1.imshow(info["average_energy"], cmap='viridis', interpolation='nearest')
            # ax1.set_title(f"{patch_labels[i]}", fontsize=8)
            # ax1.set_xlabel("i -> j")
            # ax1.set_ylabel("h -> i")
    
            ax2.imshow(info["fitted_doubly_stochastic"], cmap='viridis', interpolation='nearest')
            # ax2.set_title(f"{patch_labels[i]}", fontsize=8)
            # ax2.set_xlabel("i -> j")
            # ax2.set_ylabel("h -> i")

        logger.info(f"Singly stochastic PU recons error for NLS: {db(ss_recons_error):4f} dB")
        logger.info(f"Doubly stochastic reconstruction error CGD: {db(info['factor_error']):.4f} dB")
        logger.info(f"PU reconstruction error CGD: {db(info['average_energy_error']):.4f} dB")
        logger.info(f"Time taken CGD: {execution_time:.3f}s")
        

        assert db(info['factor_error']) <= -6, "Doubly stochastic reconstriction failed"
        assert db(info['average_energy_error']) <= -6, "PU reconstriction failed"
        assert info["is_paraunitary"], "Resulting matrix not unitary"

    except AssertionError as e:
        logger.error(f"PU reconstruction failed for {patch_labels[i]} scattering matrix of size {cur_mat_num_rows} x {cur_mat_num_rows}")
        continue



    vis.plot_poly_mats_overlay([cur_art_pu_matrix], 
                               labels=[patch_labels[i]], 
                               fs=sample_rate, 
                               title=f"{patch_labels[i]}, DS recons err ={db(info['factor_error']):.4f}dB",
                               save_path = f'{fig_path.resolve()}/{environment_name}_pu_art_matrix_{patch_labels[i]}_cgd_sa_init.png',
                              )
fig.savefig(f'{fig_path.resolve()}/{environment_name}_pu_cgd_sa_init_avg_energy_art_matrices.png')
fig2.savefig(f'{fig_path.resolve()}/{environment_name}_fitted_ds_cgd_sa_init_avg_energy_art_matrices.png')    

plt.show()

#### Convert doubly stochastic matrices to closest SA unitary matrix and observe error

In [ ]:
fig, axes = plt.subplots(num_subplots, num_subplots, figsize=(12, 12), constrained_layout=True)
art_matrix_sa_unitary_list = []

for i in range(len(art_matrix_pruned_double_stochastic)):
    try:
        cur_art_ds_matrix = art_matrix_pruned_double_stochastic[i].copy()
        cur_mat_num_rows = cur_art_ds_matrix.shape[0]
        start_time = time.perf_counter()

        cur_art_sa_unit_matrix  = closestSignAgnosticOrthonormal(np.sqrt(cur_art_ds_matrix))
        end_time = time.perf_counter()
        execution_time = end_time - start_time
        
        art_matrix_sa_unitary_list.append(cur_art_sa_unit_matrix)
        rel_err = np.linalg.norm(cur_art_ds_matrix - cur_art_sa_unit_matrix**2, ord='fro') / np.linalg.norm(cur_art_ds_matrix, ord='fro')
        ss_recons_err = np.linalg.norm(art_matrix_pruned_list[i] - cur_art_sa_unit_matrix**2, ord='fro') / np.linalg.norm(art_matrix_pruned_list[i], ord='fro')

        logger.info(f"SA Unitary SS reconstruction error: {db(ss_recons_err):.4f} dB")
        logger.info(f"SA Unitary DS reconstruction error: {db(rel_err):.4f} dB")
        logger.info(f"Time taken: {execution_time:.3f}s")

    except AssertionError as e:
        logger.error(f"SA unitary reconstruction failed for {patch_labels[i]} scattering matrix of size {cur_mat_num_rows} x {cur_mat_num_rows}")
        continue

    if i < num_subplots**2:
        ax = axes.flat[i]
        ax.imshow(art_matrix_sa_unitary_list[i]**2, cmap='viridis', interpolation='nearest')
        # ax.set_title(f"{patch_labels[i]}", fontsize=8)
        # ax.set_xlabel("i -> j")
        # ax.set_ylabel("h -> i")

plt.savefig(f'{fig_path.resolve()}/{environment_name}_sa_unitary_squared_art_matrices.png')    
plt.show()

#### Save the full PU matrix of shape 134 x 134 x num_taps and the SA unitary matrix of shape 134 x 134

In [ ]:
band_idx = 0
center_frequency = 0
save_folder = Path(f'{environment_folder}_pressure_domain_pu_reflection_matrix/')
save_folder.mkdir(parents=True, exist_ok=True)
save_path = save_folder / f'ART_kernel_band_{band_idx+1}'
art_pu_matrix_full = mm.reconstruct_full_tdart_matrix(art_matrix_pu_list_nls, 
                                                      art_patching_matrix_dense, 
                                                      patch_absorption, 
                                                      save_path=save_path, 
                                                      operate_in_pressure_domain=True)

save_folder = Path(f'{environment_folder}_pressure_domain_unitary_reflection_matrix/')
save_folder.mkdir(parents=True, exist_ok=True)
save_path = save_folder / f'ART_kernel_band_{band_idx+1}.mtx'
art_unitary_matrix_full = mm.reconstruct_full_tdart_matrix(art_matrix_sa_unitary_list, 
                                                           art_patching_matrix_dense, 
                                                           patch_absorption, 
                                                           save_path=save_path, 
                                                           operate_in_pressure_domain=True)